# SL001 — Linear Regression

**Topic ID:** SL001  
**Algorithm:** Linear Regression  
**Dataset:** California Housing (sklearn built-in)  
**Library:** scikit-learn (sklearn)  
**Goal:** Predict median house value from neighborhood features

---

## Sections
1. Imports
2. Load Dataset
3. Exploratory Data Analysis (EDA)
4. Preprocessing
5. Train Model
6. Evaluate Model
7. Visualizations
8. Coefficient Analysis
9. Summary

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries loaded successfully.')
print(f'sklearn version: {__import__("sklearn").__version__}')

## 2. Load Dataset

In [ ]:
# Load California Housing dataset
raw = fetch_california_housing(as_frame=True)
df = raw.frame

print('Dataset loaded.')
print(f'Shape: {df.shape}')  # 20640 rows × 9 columns
print(f'Target column: MedHouseVal (Median house value in $100,000 units)')
print()
print(raw.DESCR[:1200])  # Print dataset description

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# --- 3.1 Basic Info ---
print('=== Shape ===')
print(df.shape)
print()
print('=== Column Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# --- 3.2 Statistical Summary ---
df.describe()

In [ ]:
# --- 3.3 Target Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of MedHouseVal (Target)')
axes[0].set_xlabel('Median House Value ($100K)')
axes[0].set_ylabel('Count')

sns.boxplot(y=df['MedHouseVal'], ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot of MedHouseVal')
axes[1].set_ylabel('Median House Value ($100K)')

plt.tight_layout()
plt.suptitle('SL001 — Target Variable Analysis', y=1.02, fontsize=13)
plt.show()

In [ ]:
# --- 3.4 Correlation Heatmap ---
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5
)
plt.title('SL001 — Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

# Print correlation with target
print('\nCorrelation with MedHouseVal (target):')
print(corr['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False))

In [ ]:
# --- 3.5 Feature vs Target Scatter (top 4 correlated) ---
top_features = corr['MedHouseVal'].drop('MedHouseVal').abs().sort_values(ascending=False).head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df[feat], df['MedHouseVal'], alpha=0.1, s=5, color='steelblue')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('MedHouseVal')
    axes[i].set_title(f'{feat} vs MedHouseVal')

plt.suptitle('SL001 — Top 4 Features vs Target', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Preprocessing

In [ ]:
# --- 4.1 Split Features and Target ---
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'Features: {list(X.columns)}')

In [ ]:
# --- 4.2 Train/Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f'Train size: {X_train.shape[0]} rows  ({100 * len(X_train) / len(X):.0f}%)')
print(f'Test size:  {X_test.shape[0]} rows  ({100 * len(X_test) / len(X):.0f}%)')

In [ ]:
# --- 4.3 Feature Scaling (StandardScaler) ---
# Fit ONLY on training data, then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)       # transform only, do NOT fit on test

print('StandardScaler applied.')
print(f'Mean of scaled train features (should be ~0): {X_train_scaled.mean(axis=0).round(4)}')
print(f'Std  of scaled train features (should be ~1): {X_train_scaled.std(axis=0).round(4)}')

## 5. Train Model

In [ ]:
# --- 5.1 Instantiate and Train ---
model = LinearRegression(fit_intercept=True)
model.fit(X_train_scaled, y_train)

print('Model trained successfully.')
print(f'Intercept (w₀): {model.intercept_:.4f}')
print()
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)
print('Coefficients (sorted by magnitude):')
print(coef_df.to_string(index=False))

## 6. Evaluate Model

In [ ]:
# --- 6.1 Predictions ---
y_train_pred = model.predict(X_train_scaled)
y_test_pred  = model.predict(X_test_scaled)

# --- 6.2 Metrics ---
def evaluate(y_true, y_pred, label):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    print(f'--- {label} ---')
    print(f'  R²:   {r2:.4f}')
    print(f'  MAE:  {mae:.4f}')
    print(f'  MSE:  {mse:.4f}')
    print(f'  RMSE: {rmse:.4f}')
    print()
    return r2, mae, mse, rmse

train_metrics = evaluate(y_train, y_train_pred, 'TRAIN SET')
test_metrics  = evaluate(y_test,  y_test_pred,  'TEST SET')

# Baseline: always predict the mean
y_baseline = np.full_like(y_test, y_train.mean())
baseline_r2 = r2_score(y_test, y_baseline)
print(f'--- BASELINE (predict mean) ---')
print(f'  R²:  {baseline_r2:.4f}  (should be 0 or close to 0)')

## 7. Visualizations

In [ ]:
# --- 7.1 Actual vs Predicted ---
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.15, s=10, color='steelblue', label='Predictions')
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect fit')
plt.xlabel('Actual MedHouseVal')
plt.ylabel('Predicted MedHouseVal')
plt.title(f'SL001 — Actual vs Predicted  (Test R² = {test_metrics[0]:.4f})')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 7.2 Residuals Plot ---
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(y_test_pred, residuals, alpha=0.15, s=10, color='coral')
axes[0].axhline(0, color='black', linewidth=1, linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted  (check for patterns)')

# Residuals Histogram
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Residual Value')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution  (should be bell-shaped)')

plt.suptitle('SL001 — Residual Analysis', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Coefficient Analysis

In [ ]:
# --- 8.1 Coefficient Bar Chart ---
coef_df_sorted = coef_df.sort_values('Coefficient')
colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df_sorted['Coefficient']]

plt.figure(figsize=(9, 5))
plt.barh(coef_df_sorted['Feature'], coef_df_sorted['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Coefficient Value (after StandardScaler)')
plt.title('SL001 — Feature Coefficients\n(Blue = positive impact, Red = negative impact)')
plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  Positive coefficient → feature increases → house value increases')
print('  Negative coefficient → feature increases → house value decreases')
print('  Larger absolute value → stronger impact on prediction')

## 9. Summary

In [ ]:
# --- 9. Print Summary ---
print('=' * 50)
print('SL001 — Linear Regression Summary')
print('=' * 50)
print(f'Dataset:        California Housing')
print(f'Samples:        {len(df)}')
print(f'Features:       {X.shape[1]}')
print(f'Train / Test:   {len(X_train)} / {len(X_test)}')
print()
print(f'Train R²:   {train_metrics[0]:.4f}')
print(f'Test  R²:   {test_metrics[0]:.4f}')
print(f'Test  MAE:  {test_metrics[1]:.4f}  (in $100K units)')
print(f'Test  RMSE: {test_metrics[3]:.4f}  (in $100K units)')
print()
print(f'Most impactful feature (positive): {coef_df.iloc[0]["Feature"]}  ({coef_df.iloc[0]["Coefficient"]:.4f})')
print(f'Most impactful feature (negative): {coef_df.iloc[-1]["Feature"]}  ({coef_df.iloc[-1]["Coefficient"]:.4f})')
print('=' * 50)
print()
print('Next step: Update learning/SL001_linear_regression_learning.md with results.')